# Actionable Meeting Agent — Transcript → Tickets (Enterprise)

**Executive Summary (1–3 sentences)**  
Many engineering teams lose time converting meeting discussions into reproducible work items. This project implements a **multi-agent pipeline** that ingests meeting transcripts, enriches them with repo/context, extracts structured action items (owner, due date, priority), and creates tracked tickets (Jira/GitHub). Aim: reduce manual ticketing time and missed action items — demonstrated with evaluation metrics and a deployable demo.

**Key outcomes**
- Modular multi-agent architecture (Ingest → Enrich → Extract → TicketCreate).
- Memory & retrieval (vector store) for cross-meeting context and compaction summaries.
- Tool-enabled ticket creator (OpenAPI-compatible simulated Jira/GitHub).
- Observability (structured logs + metrics) and evaluation (precision/recall/F1 on extraction).

**Why enterprise cares**
- Faster ticket creation (lower PM overhead).  
- Better audit trail for decisions and follow-ups.  
- Extensible tool integrations and on-premise deployability (Docker + Compose).

**How to run (short)**  
1. Run the notebook cells in order.  
2. Or run `docker-compose up` (see Deployment section).  


## 🚀 **System Architecture Overview**  
*Component Layout + Data Flow Sequence*

```mermaid
flowchart LR

%% ================== INGEST ==================
subgraph Ingest
    A["📄 Raw Transcript (.txt / .srt)"]
end

%% ================== AGENTS ==================
subgraph Agents
    B["🧹 TranscriptNormalizer"]
    C["🧠 ContextEnricher"]
    D["📝 ActionExtractor"]
    E["🛠️ TicketCreator"]
end

%% ================== TOOLS ==================
subgraph Tools
    F["🔗 Jira / GitHub API"]
    G["🔍 Repo Search Tool"]
    H["📅 Calendar"]
end

%% ================== STORAGE ==================
subgraph Storage
    I["📚 Vector Store (FAISS / Chroma)"]
    J["⚡ Redis (state)"]
    K["🗄️ Postgres / JSON Store"]
end

%% ================== DATA FLOWS ==================
A --> B --> C --> D --> E
C -->|Repo Context| G
D -->|Actions JSON| I
E --> F
D --> J
I --> C

%% ================== STYLES ==================
%% Agents
style B fill:#e8c2ff,stroke:#333,stroke-width:1px,stroke-dasharray: 3 3
style C fill:#d0e8ff,stroke:#333,stroke-width:1px,stroke-dasharray: 3 3
style D fill:#fff6a5,stroke:#333,stroke-width:1px,stroke-dasharray: 3 3
style E fill:#ffd1a5,stroke:#333,stroke-width:1px,stroke-dasharray: 3 3

%% Storage
style I fill:#ccffcc,stroke:#333,stroke-width:1px
style J fill:#ffcccc,stroke:#333,stroke-width:1px
style K fill:#f0f0f0,stroke:#333,stroke-width:1px

%% Tools
classDef toolNode fill:#fff,stroke:#bbb,stroke-width:1px,stroke-dasharray: 2 2;
class F,G,H toolNode;


In [4]:
# run this cell to install dependencies 
!pip install prometheus_client sentence-transformers faiss-cpu transformers[torch] --quiet


In [18]:
import json
import logging
import uuid
from typing import List, Dict, Any
from prometheus_client import Counter, Gauge

# Setup structured JSON logger
logger = logging.getLogger("observability")
logger.setLevel(logging.INFO)
fh = logging.FileHandler("observability.log")
formatter = logging.Formatter('%(message)s')
fh.setFormatter(formatter)
logger.addHandler(fh)

# Metrics (basic, exported later)
ACTIONS_PROCESSED = Counter('actions_processed_total', 'Total action items processed')
EXTRACTION_CONF_SUM = Gauge('extraction_confidence_sum', 'Sum of extractor confidences')

# --- Agent stubs ---
class TranscriptNormalizer:
    """Cleans transcript text, tag speakers, do simple segmentation."""
    def normalize(self, raw_transcript: str) -> Dict[str, Any]:
        # Minimal normalization: split by lines, strip, attach ids
        lines = [ln.strip() for ln in raw_transcript.splitlines() if ln.strip()]
        normalized = {"segments": [{"id": i, "text": lines[i]} for i in range(len(lines))], "meta": {"len": len(lines)}}
        logger.info(json.dumps({"event":"normalize", "segments": len(lines)}))
        return normalized

class ContextEnricher:
    """Adds repo / meeting history context (placeholder)."""
    def __init__(self, repo_search_fn=None, memory_retrieval_fn=None):
        self.repo_search_fn = repo_search_fn
        self.memory_retrieval_fn = memory_retrieval_fn

    def enrich(self, normalized_transcript: Dict[str, Any]) -> Dict[str, Any]:
        # basic enrichment: ask repo_search_fn for top matches for the whole transcript (if available)
        text = " ".join(seg["text"] for seg in normalized_transcript["segments"][:10])
        repo_matches = []
        if self.repo_search_fn:
            repo_matches = self.repo_search_fn(text)
        memory_ctx = []
        if self.memory_retrieval_fn:
            memory_ctx = self.memory_retrieval_fn(text)
        enriched = {"normalized": normalized_transcript, "repo_matches": repo_matches, "memory_ctx": memory_ctx}
        logger.info(json.dumps({"event":"enrich", "repo_hits": len(repo_matches), "memory_hits": len(memory_ctx)}))
        return enriched

class ActionExtractor:
    """Extracts structured actions from enriched transcript. Returns list of action dicts."""
    def __init__(self, model=None):
        # model can be an LLM wrapper or heuristic; for now we use a simple heuristic extractor
        self.model = model

    def extract(self, enriched: Dict[str, Any]) -> List[Dict[str, Any]]:
        segments = enriched["normalized"]["segments"]
        actions = []
        for seg in segments:
            text = seg["text"]
            # VERY simple heuristics for demo: lines that start with verbs or contain 'TODO' or 'action'
            if text.lower().startswith(("todo", "action", "assign", "please")) or "todo" in text.lower() or "action item" in text.lower():
                a = {
                    "id": str(uuid.uuid4()),
                    "text": text,
                    "owner": None,
                    "due": None,
                    "confidence": 0.6,  # heuristic confidence
                    "source_segment_id": seg["id"]
                }
                actions.append(a)
                ACTIONS_PROCESSED.inc()
                EXTRACTION_CONF_SUM.set(EXTRACTION_CONF_SUM._value.get() + a["confidence"] if hasattr(EXTRACTION_CONF_SUM, "_value") else a["confidence"])
        logger.info(json.dumps({"event":"extract", "actions_found": len(actions)}))
        return actions

# Simple controller orchestration
class Controller:
    def __init__(self, normalizer, enricher, extractor, ticket_creator_fn=None):
        self.normalizer = normalizer
        self.enricher = enricher
        self.extractor = extractor
        self.ticket_creator_fn = ticket_creator_fn

    def run(self, raw_transcript: str, create_tickets: bool=False):
        norm = self.normalizer.normalize(raw_transcript)
        enriched = self.enricher.enrich(norm)
        actions = self.extractor.extract(enriched)

        # structured output
        result = {"normalized": norm, "enriched": {"repo_matches": enriched["repo_matches"], "memory_ctx_count": len(enriched["memory_ctx"])}, "actions": actions}
        logger.info(json.dumps({"event":"controller_complete", "actions": len(actions)}))
        if create_tickets and self.ticket_creator_fn and actions:
            tickets = []
            for a in actions:
                t = self.ticket_creator_fn(a)  # simulated ticket create
                tickets.append(t)
            result["tickets"] = tickets
            logger.info(json.dumps({"event":"tickets_created", "count": len(tickets)}))
        return result

# Simple ticket creator (simulated) — will be replaced by tools server in next step
def simulated_jira_create(action):
    ticket_id = "SIM-"+str(uuid.uuid4())[:8]
    ticket_url = f"https://sim-jira.local/browse/{ticket_id}"
    logger.info(json.dumps({"event":"jira_create", "action_id": action["id"], "ticket": ticket_id}))
    return {"ticket_id": ticket_id, "url": ticket_url, "action_id": action["id"]}

# Example usage
if __name__ == "__main__":
    raw = """Alice: TODO - Add unit tests for PaymentService
Bob: We should assign someone to integrate the new auth flow.
Carol: Action item: prepare migration script by next Wednesday.
Note: This is not an action, just FYI."""
    ctrl = Controller(TranscriptNormalizer(), ContextEnricher(), ActionExtractor(), simulated_jira_create)
    out = ctrl.run(raw, create_tickets=True)
    print(json.dumps(out, indent=2))


ValueError: Duplicated timeseries in CollectorRegistry: {'actions_processed_total', 'actions_processed', 'actions_processed_created'}

In [2]:
import json
import logging
import uuid
from typing import List, Dict, Any
from prometheus_client import Counter, Gauge

# Setup structured JSON logger
logger = logging.getLogger("observability")
logger.setLevel(logging.INFO)
fh = logging.FileHandler("observability.log")
formatter = logging.Formatter('%(message)s')
fh.setFormatter(formatter)
logger.addHandler(fh)

# Metrics (basic, exported later)
ACTIONS_PROCESSED = Counter('actions_processed_total', 'Total action items processed')
EXTRACTION_CONF_SUM = Gauge('extraction_confidence_sum', 'Sum of extractor confidences')

# --- Agent stubs ---
class TranscriptNormalizer:
    """Cleans transcript text, tag speakers, do simple segmentation."""
    def normalize(self, raw_transcript: str) -> Dict[str, Any]:
        # Minimal normalization: split by lines, strip, attach ids
        lines = [ln.strip() for ln in raw_transcript.splitlines() if ln.strip()]
        normalized = {"segments": [{"id": i, "text": lines[i]} for i in range(len(lines))], "meta": {"len": len(lines)}}
        logger.info(json.dumps({"event":"normalize", "segments": len(lines)}))
        return normalized

class ContextEnricher:
    """Adds repo / meeting history context (placeholder)."""
    def __init__(self, repo_search_fn=None, memory_retrieval_fn=None):
        self.repo_search_fn = repo_search_fn
        self.memory_retrieval_fn = memory_retrieval_fn

    def enrich(self, normalized_transcript: Dict[str, Any]) -> Dict[str, Any]:
        # basic enrichment: ask repo_search_fn for top matches for the whole transcript (if available)
        text = " ".join(seg["text"] for seg in normalized_transcript["segments"][:10])
        repo_matches = []
        if self.repo_search_fn:
            repo_matches = self.repo_search_fn(text)
        memory_ctx = []
        if self.memory_retrieval_fn:
            memory_ctx = self.memory_retrieval_fn(text)
        enriched = {"normalized": normalized_transcript, "repo_matches": repo_matches, "memory_ctx": memory_ctx}
        logger.info(json.dumps({"event":"enrich", "repo_hits": len(repo_matches), "memory_hits": len(memory_ctx)}))
        return enriched

class ActionExtractor:
    """Extracts structured actions from enriched transcript. Returns list of action dicts."""
    def __init__(self, model=None):
        # model can be an LLM wrapper or heuristic; for now we use a simple heuristic extractor
        self.model = model

    def extract(self, enriched: Dict[str, Any]) -> List[Dict[str, Any]]:
        segments = enriched["normalized"]["segments"]
        actions = []
        for seg in segments:
            text = seg["text"]
            # VERY simple heuristics for demo: lines that start with verbs or contain 'TODO' or 'action'
            if text.lower().startswith(("todo", "action", "assign", "please")) or "todo" in text.lower() or "action item" in text.lower():
                a = {
                    "id": str(uuid.uuid4()),
                    "text": text,
                    "owner": None,
                    "due": None,
                    "confidence": 0.6,  # heuristic confidence
                    "source_segment_id": seg["id"]
                }
                actions.append(a)
                ACTIONS_PROCESSED.inc()
                EXTRACTION_CONF_SUM.set(EXTRACTION_CONF_SUM._value.get() + a["confidence"] if hasattr(EXTRACTION_CONF_SUM, "_value") else a["confidence"])
        logger.info(json.dumps({"event":"extract", "actions_found": len(actions)}))
        return actions

# Simple controller orchestration
class Controller:
    def __init__(self, normalizer, enricher, extractor, ticket_creator_fn=None):
        self.normalizer = normalizer
        self.enricher = enricher
        self.extractor = extractor
        self.ticket_creator_fn = ticket_creator_fn

    def run(self, raw_transcript: str, create_tickets: bool=False):
        norm = self.normalizer.normalize(raw_transcript)
        enriched = self.enricher.enrich(norm)
        actions = self.extractor.extract(enriched)

        # structured output
        result = {"normalized": norm, "enriched": {"repo_matches": enriched["repo_matches"], "memory_ctx_count": len(enriched["memory_ctx"])}, "actions": actions}
        logger.info(json.dumps({"event":"controller_complete", "actions": len(actions)}))
        if create_tickets and self.ticket_creator_fn and actions:
            tickets = []
            for a in actions:
                t = self.ticket_creator_fn(a)  # simulated ticket create
                tickets.append(t)
            result["tickets"] = tickets
            logger.info(json.dumps({"event":"tickets_created", "count": len(tickets)}))
        return result

# Simple ticket creator (simulated) — will be replaced by tools server in next step
def simulated_jira_create(action):
    ticket_id = "SIM-"+str(uuid.uuid4())[:8]
    ticket_url = f"https://sim-jira.local/browse/{ticket_id}"
    logger.info(json.dumps({"event":"jira_create", "action_id": action["id"], "ticket": ticket_id}))
    return {"ticket_id": ticket_id, "url": ticket_url, "action_id": action["id"]}

# Example usage
if __name__ == "__main__":
    raw = """Alice: TODO - Add unit tests for PaymentService
Bob: We should assign someone to integrate the new auth flow.
Carol: Action item: prepare migration script by next Wednesday.
Note: This is not an action, just FYI."""
    ctrl = Controller(TranscriptNormalizer(), ContextEnricher(), ActionExtractor(), simulated_jira_create)
    out = ctrl.run(raw, create_tickets=True)
    print(json.dumps(out, indent=2))


{
  "normalized": {
    "segments": [
      {
        "id": 0,
        "text": "Alice: TODO - Add unit tests for PaymentService"
      },
      {
        "id": 1,
        "text": "Bob: We should assign someone to integrate the new auth flow."
      },
      {
        "id": 2,
        "text": "Carol: Action item: prepare migration script by next Wednesday."
      },
      {
        "id": 3,
        "text": "Note: This is not an action, just FYI."
      }
    ],
    "meta": {
      "len": 4
    }
  },
  "enriched": {
    "repo_matches": [],
    "memory_ctx_count": 0
  },
  "actions": [
    {
      "id": "bebd66aa-87c7-4292-98fb-3c3d0116908c",
      "text": "Alice: TODO - Add unit tests for PaymentService",
      "owner": null,
      "due": null,
      "confidence": 0.6,
      "source_segment_id": 0
    },
    {
      "id": "f95bf9ac-b8ed-4e95-93fb-9c92771776d9",
      "text": "Carol: Action item: prepare migration script by next Wednesday.",
      "owner": null,
      "due": null,
      

In [7]:
pip install tf-keras

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------------------------------ --------- 1.3/1.7 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 4.6 MB/s  0:00:00
   ---------------------------------------- 0.0/331.9 MB ? eta -:--:--
   ---------------------------------------- 1.3/331.9 MB 5.6 MB/s eta 0:00:59
   ---------------------------------------- 2.6/331.9 MB 5.8 MB/s eta 0:00:57
    --------------------------------------- 4.2/331.9 MB 6.3 MB/s eta 0:00:53
    --------------------------------------- 5.5/331.9 MB 6.3 MB/s eta 0:00:52
    --------------------------------------- 7.1/331.9 MB 6.5 MB/s eta 0:00:50
   - -------------------------------------- 8.7/331.9 MB 6.5 MB/s eta 0:00:50
   - -------------------------------------- 10.2/331.9 MB 6.5 MB/s eta 0:00:50
   - -------------------------------------- 11.5/331.9 MB 6.5 MB/s eta 0:00:50
   - -----------

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\2k22c\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\tensorflow\\compiler\\mlir\\quantization\\tensorflow\\python\\pywrap_function_lib.pyd'
Consider using the `--user` option or check the permissions.



In [17]:
# Memory & Context Engineering: FAISS + Sentence-Transformers + simple compaction
import os
import re
import time
import json
import uuid
from typing import List, Dict, Any, Tuple
import numpy as np

# embeddings model
from sentence_transformers import SentenceTransformer
import faiss

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # small and fast
embedder = SentenceTransformer(MODEL_NAME)
EMB_DIM = embedder.get_sentence_embedding_dimension()

# MemoryStore using FAISS (in-memory for notebook demo)
class MemoryStore:
    def __init__(self, dim: int = EMB_DIM):
        self.dim = dim
        # faiss index (Flat L2)
        self.index = faiss.IndexFlatL2(dim)
        self.metadatas: List[Dict[str, Any]] = []  # parallel list to index vectors
        self.embeddings = []  # keep embeddings so we can rebuild index if needed

    def _text_to_embedding(self, text: str) -> np.ndarray:
        emb = embedder.encode([text], show_progress_bar=False, convert_to_numpy=True)
        return emb[0]

    def add_meeting(self, meeting_id: str, text: str, meta: Dict[str, Any]=None):
        """Add a meeting text as a single memory record (summary may be applied later)."""
        if meta is None:
            meta = {}
        timestamp = int(time.time())
        embedding = self._text_to_embedding(text)
        self.index.add(np.array([embedding]).astype('float32'))
        self.embeddings.append(embedding)
        self.metadatas.append({"meeting_id": meeting_id, "text": text, "meta": meta, "ts": timestamp})
        return {"meeting_id": meeting_id}

    def query(self, query_text: str, k: int = 3) -> List[Dict[str, Any]]:
        """Return top-k memories as list of {meeting_id, text, score, meta}"""
        if len(self.metadatas) == 0:
            return []
        q_emb = self._text_to_embedding(query_text).astype('float32').reshape(1, -1)
        D, I = self.index.search(q_emb, k)
        results = []
        for dist, idx in zip(D[0], I[0]):
            if idx < 0 or idx >= len(self.metadatas):
                continue
            md = self.metadatas[idx]
            results.append({"meeting_id": md["meeting_id"], "text": md["text"], "score": float(dist), "meta": md.get("meta", {})})
        return results

    def compact_old_meetings(self, keep_top_n: int = 5):
        """
        Compact older meetings into 1-2 sentence summaries if total meetings exceed keep_top_n.
        Strategy: pick oldest meetings beyond keep_top_n, summarize by extractive method: 
        extract few representative sentences using sentence embeddings (centroid nearest).
        Replace each long meeting text with the compacted summary and reindex.
        """
        total = len(self.metadatas)
        if total <= keep_top_n:
            return {"compacted": 0, "total": total}
        # sort by timestamp
        sorted_idx = sorted(range(total), key=lambda i: self.metadatas[i]["ts"])
        to_compact = sorted_idx[: max(0, total - keep_top_n)]
        compacted_count = 0
        for idx in to_compact:
            md = self.metadatas[idx]
            orig_text = md["text"]
            summary = self._extractive_summary(orig_text, n_sentences=2)
            # replace text and re-create embedding
            new_emb = self._text_to_embedding(summary)
            # update embedding arrays and metadata
            self.metadatas[idx]["text"] = summary
            self.embeddings[idx] = new_emb
            compacted_count += 1
        # rebuild FAISS index from updated embeddings for consistency
        emb_arr = np.array(self.embeddings).astype('float32')
        self.index = faiss.IndexFlatL2(self.dim)
        if len(emb_arr) > 0:
            self.index.add(emb_arr)
        return {"compacted": compacted_count, "total": total}

    def _extractive_summary(self, text: str, n_sentences: int = 2) -> str:
        """
        Simple extractive summarizer:
        - Split into sentences
        - Embed each sentence
        - Compute centroid and pick sentences nearest to centroid
        """
        # naive sentence split (works ok for meeting-style text)
        sents = re.split(r'(?<=[.!?])\s+', text.strip())
        if len(sents) <= n_sentences:
            return text.strip()
        sent_embs = embedder.encode(sents, show_progress_bar=False, convert_to_numpy=True)
        centroid = np.mean(sent_embs, axis=0, keepdims=True)
        # compute L2 distances
        dists = np.linalg.norm(sent_embs - centroid, axis=1)
        top_idx = np.argsort(dists)[:n_sentences]
        chosen = [sents[i].strip() for i in sorted(top_idx)]
        return " ".join(chosen)

    def dump_index_summary(self):
        """Return a small human-readable summary of stored memories."""
        return [{"meeting_id": md["meeting_id"], "text_preview": md["text"][:200], "ts": md["ts"]} for md in self.metadatas]


# --- Demo: create memory store, add two meetings, query, compact ---
memory = MemoryStore()

m1 = """Meeting: Payment migration
Alice: Decision - we will migrate PaymentService to v2 next sprint.
Bob: Action: create migration plan and assign to backend team. TODO by Fri.
Alice: Note - update docs and notify security team.
"""
m2 = """Meeting: Auth flow
Carol: Action item: evaluate new SSO provider, deadline next month.
Bob: We discussed rolling auth change with feature flag.
Decisions: postpone full rollout until after testing.
"""

print("Adding two meetings to memory store...")
memory.add_meeting("meeting-001", m1, meta={"title":"Payment migration"})
memory.add_meeting("meeting-002", m2, meta={"title":"Auth flow"})

print("Memory index summary:")
print(json.dumps(memory.dump_index_summary(), indent=2))

print("\nQuerying memory for 'migration plan and decisions about payment service':")
res = memory.query("migration plan decisions about payment service", k=3)
print(json.dumps(res, indent=2))

print("\nNow demonstrate compaction: adding more meetings (simulated) to push compaction...")
# add a few filler meetings to trigger compaction
for i in range(3):
    txt = f"Old meeting {i}: some long details about topic {i}. Decision: store as historical note. Follow up later."
    memory.add_meeting(f"meeting-old-{i}", txt, meta={"title": f"old-{i}"})

print("Before compaction total:", len(memory.metadatas))
res_compact = memory.compact_old_meetings(keep_top_n=4)
print("Compaction result:", res_compact)
print("After compaction index summary:")
print(json.dumps(memory.dump_index_summary(), indent=2))


ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [9]:
# Tool Server: FastAPI (simulated Jira, Repo Search, Calendar)

import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import uuid
import threading
import time

app = FastAPI(
    title="Actionable Meeting Agent Tools API",
    description="Simulated Jira, Repo Search, and Calendar tools with OpenAPI specs.",
    version="1.0.0"
)

# -----------------------------
# Request/Response Models
# -----------------------------
class JiraCreateRequest(BaseModel):
    action_id: str
    title: str
    description: str = ""
    owner: str = None
    due: str = None

class JiraCreateResponse(BaseModel):
    ticket_id: str
    url: str
    action_id: str

class RepoSearchRequest(BaseModel):
    query: str
    top_k: int = 3

class RepoSearchResponse(BaseModel):
    results: list

class CalendarCreateRequest(BaseModel):
    title: str
    start: str
    end: str
    attendees: list = []

class CalendarCreateResponse(BaseModel):
    event_id: str
    link: str

# -----------------------------
# Endpoints
# -----------------------------
@app.post("/jira/create", response_model=JiraCreateResponse)
def create_jira_ticket(req: JiraCreateRequest):
    ticket_id = f"JIRA-{str(uuid.uuid4())[:8]}"
    url = f"https://sim-jira.local/browse/{ticket_id}"
    return JiraCreateResponse(ticket_id=ticket_id, url=url, action_id=req.action_id)

@app.post("/repo/search", response_model=RepoSearchResponse)
def repo_search(req: RepoSearchRequest):
    # Simulated repo matches
    fake_results = [
        {"file": "src/payment/payment_service.py", "snippet": "def process_payment(): ..."},
        {"file": "src/auth/auth_flow.js", "snippet": "function authenticate() {...}"},
        {"file": "docs/migration_guide.md", "snippet": "Step 1: Update config..."}
    ]
    return RepoSearchResponse(results=fake_results[: req.top_k])

@app.post("/calendar/create", response_model=CalendarCreateResponse)
def calendar_create(req: CalendarCreateRequest):
    event_id = f"EVT-{str(uuid.uuid4())[:8]}"
    link = f"https://sim-calendar.local/event/{event_id}"
    return CalendarCreateResponse(event_id=event_id, link=link)

# -----------------------------
# Run server in background thread
# -----------------------------
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

time.sleep(2)
print("🚀 Tool Server running at http://127.0.0.1:8000")
print("📘 OpenAPI docs available at http://127.0.0.1:8000/docs")


INFO:     Started server process [18948]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 Tool Server running at http://127.0.0.1:8000
📘 OpenAPI docs available at http://127.0.0.1:8000/docs


INFO:     127.0.0.1:58585 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:58585 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:58585 - "GET /openapi.json HTTP/1.1" 200 OK


In [16]:
import requests
from pprint import pprint

TOOL_SERVER = "http://127.0.0.1:8000"

def jira_tool_create(action: dict) -> dict:
    # normalize payload: avoid sending JSON nulls that might trigger validation errors
    payload = {
        "action_id": action.get("id", ""),
        "title": action.get("text", "")[:240],   # truncate long titles
        "description": action.get("text", "") or "",
        "owner": action.get("owner") or "",      # empty string instead of None
        "due": action.get("due") or ""
    }
    try:
        r = requests.post(f"{TOOL_SERVER}/jira/create", json=payload, timeout=5)
        # if validation error, print server response body to help debug
        if r.status_code != 200 and r.status_code != 201:
            print("jira_tool_create - non-200 response:", r.status_code)
            print("Response body:", r.text)
            r.raise_for_status()
        return r.json()
    except requests.exceptions.HTTPError as e:
        print("jira_tool_create HTTPError:", e)
        # log full response if available
        try:
            print("Server response:", r.status_code, r.text)
        except Exception:
            pass
        # fallback simulated ticket
        return {"ticket_id": f"SIM-FALLBACK-{action['id'][:8]}", "url": None, "action_id": action["id"]}
    except Exception as e:
        print("jira_tool_create error:", e)
        return {"ticket_id": f"SIM-FALLBACK-{action['id'][:8]}", "url": None, "action_id": action["id"]}

def repo_search_tool(query: str, top_k: int = 3) -> list:
    payload = {"query": query, "top_k": top_k}
    try:
        r = requests.post(f"{TOOL_SERVER}/repo/search", json=payload, timeout=5)
        r.raise_for_status()
        return r.json().get("results", [])
    except Exception as e:
        print("repo_search_tool error:", e)
        return []

# reinstantiate enricher/controller to ensure we use repo + memory + new jira tool
enricher = ContextEnricher(repo_search_fn=repo_search_tool, memory_retrieval_fn=memory.query)
ctrl = Controller(
    normalizer=TranscriptNormalizer(),
    enricher=enricher,
    extractor=ActionExtractor(),
    ticket_creator_fn=jira_tool_create
)

out = ctrl.run(raw, create_tickets=True)
print("Controller output (trimmed):")
print("Actions found:", len(out.get("actions", [])))
if out.get("tickets"):
    pprint(out["tickets"])
else:
    print("No tickets created.")

NameError: name 'memory' is not defined

In [12]:
from prometheus_client import CollectorRegistry, start_http_server, Histogram, Counter, Gauge
import time

# Create a separate registry
registry = CollectorRegistry()

# Metrics using this registry
EXTRACTION_CONF_HIST = Histogram(
    'extraction_confidence_hist',
    'Histogram of extraction confidences',
    buckets=[0.1,0.3,0.5,0.7,0.9,1.0],
    registry=registry
)
TOOL_ERRORS = Counter('tool_errors_total', 'Number of tool invocation errors', registry=registry)
LAST_RUN_TS = Gauge('last_controller_run_timestamp', 'Unix timestamp of last controller run', registry=registry)

# Start HTTP server with this registry
start_http_server(8001, registry=registry)
print("Prometheus metrics server started at http://127.0.0.1:8001/metrics")


Prometheus metrics server started at http://127.0.0.1:8001/metrics


In [ ]:
!pip install opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp --quiet

In [19]:
# Step 5B — Lightweight OpenTelemetry tracing (Console exporter)

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

provider = TracerProvider()
processor = SimpleSpanProcessor(ConsoleSpanExporter())
provider.add_span_processor(processor)
trace.set_tracer_provider(provider)
tracer = trace.get_tracer(__name__)

# Example: add tracing to Controller.run and agent methods (monkeypatch simple wrappers)
orig_run = Controller.run

def traced_run(self, raw_transcript, create_tickets=False):
    with tracer.start_as_current_span("controller.run") as span:
        span.set_attribute("create_tickets", bool(create_tickets))
        res = orig_run(self, raw_transcript, create_tickets=create_tickets)
        span.set_attribute("actions_count", len(res.get("actions", [])))
        return res

Controller.run = traced_run

print("OpenTelemetry tracing enabled (console exporter). Run the controller to see traces printed.")


Overriding of current TracerProvider is not allowed


OpenTelemetry tracing enabled (console exporter). Run the controller to see traces printed.


In [20]:
# --- Update jira client to increment TOOL_ERRORS on failures, run controller, and observe metrics ---
import requests, time
from pprint import pprint

# update jira function to record tool error metric
def jira_tool_create_with_metrics(action: dict) -> dict:
    payload = {
        "action_id": action.get("id", ""),
        "title": (action.get("text", "") or "")[:240],
        "description": action.get("text", "") or "",
        "owner": action.get("owner") or "",
        "due": action.get("due") or ""
    }
    try:
        r = requests.post(f"{TOOL_SERVER}/jira/create", json=payload, timeout=5)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print("jira_tool_create_with_metrics error:", e)
        try:
            TOOL_ERRORS.inc()
        except Exception:
            print("Could not increment TOOL_ERRORS metric (not defined?)")
        return {"ticket_id": f"SIM-FALLBACK-{action['id'][:8]}", "url": None, "action_id": action["id"]}

# re-create enricher/controller with the new jira client
enricher = ContextEnricher(repo_search_fn=repo_search_tool, memory_retrieval_fn=memory.query)
ctrl = Controller(
    normalizer=TranscriptNormalizer(),
    enricher=enricher,
    extractor=ActionExtractor(),
    ticket_creator_fn=jira_tool_create_with_metrics
)

# run controller
out = ctrl.run(raw, create_tickets=True)
print("\nController run complete. Actions found:", len(out.get("actions", [])))

# observe metrics after run
observe_after_run(out.get("actions", []))
print("Called observe_after_run(...) to record confidences and update last run timestamp.")

# small wait to ensure metrics are exported
time.sleep(0.5)

# fetch and print the key metric lines from metrics endpoint to verify
try:
    r = requests.get("http://127.0.0.1:8001/metrics", timeout=3)
    txt = r.text
    # filter for a few interesting metrics
    interesting = []
    for line in txt.splitlines():
        if line.startswith("extraction_confidence_hist") or line.startswith("actions_processed_total") or line.startswith("tool_errors_total") or line.startswith("last_controller_run_timestamp"):
            interesting.append(line)
    print("\n--- Selected metrics ---")
    print("\n".join(interesting))
except Exception as e:
    print("Could not fetch /metrics:", e)


NameError: name 'memory' is not defined

In [15]:
!pip install fakeredis==1.7.0 --quiet


In [46]:
import fakeredis
import json
import time
from typing import Dict, Any, List

# --- Setup fakeredis (in-memory redis-like store) ---
redis_client = fakeredis.FakeStrictRedis()

class RedisApprovalStore:
    """
    Simple Redis-based approval store.
    Keys:
      approvals:<id> -> JSON payload {action, status, created_ts, ticket_info?}
      approvals:list -> list or set of pending approval ids (we keep as Redis list)
    """
    def __init__(self, client):
        self.client = client
        self.list_key = "approvals:list"

    def store_pending(self, action: Dict[str, Any]) -> str:
        approval_id = f"appr-{str(uuid.uuid4())[:8]}"
        payload = {
            "approval_id": approval_id,
            "action": action,
            "status": "pending",
            "created_ts": int(time.time()),
            "ticket": None,
            "notes": None
        }
        self.client.set(f"approvals:{approval_id}", json.dumps(payload))
        # push to list (left push so latest at head)
        self.client.lpush(self.list_key, approval_id)
        return approval_id

    def list_pending(self, limit: int = 50) -> List[Dict[str, Any]]:
        ids = self.client.lrange(self.list_key, 0, limit - 1)
        results = []
        for aid in ids:
            key = f"approvals:{aid.decode() if isinstance(aid, bytes) else aid}"
            raw = self.client.get(key)
            if not raw:
                continue
            payload = json.loads(raw)
            results.append(payload)
        return results

    def get(self, approval_id: str) -> Dict[str, Any]:
        raw = self.client.get(f"approvals:{approval_id}")
        if not raw:
            return {}
        return json.loads(raw)

    def update(self, approval_id: str, patch: Dict[str, Any]):
        key = f"approvals:{approval_id}"
        raw = self.client.get(key)
        if not raw:
            return None
        payload = json.loads(raw)
        payload.update(patch)
        self.client.set(key, json.dumps(payload))
        # if status changed to non-pending, remove from pending list
        if payload.get("status") != "pending":
            try:
                # remove all occurrences of this id from list
                self.client.lrem(self.list_key, 0, approval_id)
            except Exception:
                pass
        return payload

# instantiate store
approval_store = RedisApprovalStore(redis_client)

# --- Controller helpers: store pending instead of creating tickets ---
def controller_store_pending(controller: Controller, raw_transcript: str) -> Dict[str, Any]:
    """
    Run normalization, enrichment, extraction, but DO NOT create tickets.
    Instead, store each extracted action into Redis as pending approval.
    Returns the list of created approval ids.
    """
    norm = controller.normalizer.normalize(raw_transcript)
    enriched = controller.enricher.enrich(norm)
    actions = controller.extractor.extract(enriched)
    approval_ids = []
    for a in actions:
        aid = approval_store.store_pending(a)
        approval_ids.append(aid)
        logger.info(json.dumps({"event":"approval_stored", "approval_id": aid, "action_id": a["id"]}))
    return {"approvals_created": approval_ids, "count": len(approval_ids)}

# helper to list pending approvals
def list_pending_approvals():
    pending = approval_store.list_pending()
    # print readable summary
    print(f"Pending approvals: {len(pending)}")
    for p in pending:
        print("-"*40)
        print("approval_id:", p["approval_id"])
        print("status:", p["status"])
        print("action_text:", p["action"]["text"])
        print("created_ts:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(p["created_ts"])))
    return pending

# helper to approve (calls jira tool and updates store)
def approve_and_create(approval_id: str, ticket_creator_fn):
    payload = approval_store.get(approval_id)
    if not payload:
        print("Approval ID not found:", approval_id)
        return None
    if payload["status"] != "pending":
        print("Approval is not pending (status=%s)" % payload["status"])
        return payload
    action = payload["action"]
    # call the ticket tool
    ticket = ticket_creator_fn(action)
    patch = {"status": "created", "ticket": ticket, "approved_ts": int(time.time())}
    updated = approval_store.update(approval_id, patch)
    logger.info(json.dumps({"event":"approval_approved", "approval_id": approval_id, "ticket": ticket}))
    return updated

# helper to defer (postpone)
def defer_approval(approval_id: str, reason: str = None):
    payload = approval_store.get(approval_id)
    if not payload:
        print("Approval ID not found:", approval_id)
        return None
    patch = {"status": "deferred", "notes": reason, "deferred_ts": int(time.time())}
    updated = approval_store.update(approval_id, patch)
    logger.info(json.dumps({"event":"approval_deferred", "approval_id": approval_id, "reason": reason}))
    return updated

print("Approval store ready. Use controller_store_pending(controller, raw_transcript) to create pending approvals.")


Approval store ready. Use controller_store_pending(controller, raw_transcript) to create pending approvals.


In [47]:
# Create pending approvals (pause)
res = controller_store_pending(ctrl, raw)   # ctrl from earlier
print("Stored approvals:", res)

# List them
pending = list_pending_approvals()


INFO:observability:{"event": "normalize", "segments": 4}


INFO:     127.0.0.1:47136 - "POST /repo/search HTTP/1.1" 200 OK


INFO:observability:{"event": "enrich", "repo_hits": 3, "memory_hits": 3}
INFO:observability:{"event": "extract", "actions_found": 2}
INFO:observability:{"event": "approval_stored", "approval_id": "appr-9718c748", "action_id": "1792a842-4c00-491c-9966-fd9d32d0a01f"}
INFO:observability:{"event": "approval_stored", "approval_id": "appr-0f0e4743", "action_id": "f0a943e3-1cfc-449f-8e48-5e37a0960a29"}


Stored approvals: {'approvals_created': ['appr-9718c748', 'appr-0f0e4743'], 'count': 2}
Pending approvals: 2
----------------------------------------
approval_id: appr-0f0e4743
status: pending
action_text: Carol: Action item: prepare migration script by next Wednesday.
created_ts: 2025-11-16 01:45:56
----------------------------------------
approval_id: appr-9718c748
status: pending
action_text: Alice: TODO - Add unit tests for PaymentService
created_ts: 2025-11-16 01:45:56


In [ ]:
# Create pending approvals (pause)
res = controller_store_pending(ctrl, raw)   # ctrl from earlier
print("Stored approvals:", res)

# List them
pending = list_pending_approvals()


INFO:observability:{"event": "normalize", "segments": 4}


INFO:     127.0.0.1:40544 - "POST /repo/search HTTP/1.1" 200 OK


INFO:observability:{"event": "enrich", "repo_hits": 3, "memory_hits": 3}
INFO:observability:{"event": "extract", "actions_found": 2}
INFO:observability:{"event": "approval_stored", "approval_id": "appr-53ba8eda", "action_id": "62b69539-f1cd-44f7-af4a-33383b34c3af"}
INFO:observability:{"event": "approval_stored", "approval_id": "appr-967a086e", "action_id": "452b2f39-6bea-4453-b6d0-f5a203f0e014"}


Stored approvals: {'approvals_created': ['appr-53ba8eda', 'appr-967a086e'], 'count': 2}
Pending approvals: 4
----------------------------------------
approval_id: appr-967a086e
status: pending
action_text: Carol: Action item: prepare migration script by next Wednesday.
created_ts: 2025-11-16 01:46:02
----------------------------------------
approval_id: appr-53ba8eda
status: pending
action_text: Alice: TODO - Add unit tests for PaymentService
created_ts: 2025-11-16 01:46:02
----------------------------------------
approval_id: appr-0f0e4743
status: pending
action_text: Carol: Action item: prepare migration script by next Wednesday.
created_ts: 2025-11-16 01:45:56
----------------------------------------
approval_id: appr-9718c748
status: pending
action_text: Alice: TODO - Add unit tests for PaymentService
created_ts: 2025-11-16 01:45:56


In [ ]:
try:
    from ipywidgets import VBox, HBox, Button, Output, Label
    from IPython.display import display, clear_output

    outw = Output()

    def refresh_ui(_=None):
        clear_output(wait=True)
        pending = approval_store.list_pending()
        if not pending:
            display(Label("No pending approvals"))
            return
        rows = []
        for p in pending:
            lbl = Label(f"{p['approval_id']} - {p['action']['text'][:120]}")
            btn_app = Button(description="Approve", button_style='success')
            btn_def = Button(description="Defer", button_style='warning')
            def on_approve(b, aid=p['approval_id']):
                with outw:
                    print("Approving", aid)
                    res = approve_and_create(aid, jira_tool_create_with_metrics)
                    print("Approved:", res)
                    refresh_ui()
            def on_defer(b, aid=p['approval_id']):
                with outw:
                    print("Deferring", aid)
                    res = defer_approval(aid, reason="Deferred via UI")
                    print("Deferred:", res)
                    refresh_ui()
            btn_app.on_click(on_approve)
            btn_def.on_click(on_defer)
            rows.append(HBox([lbl, btn_app, btn_def]))
        display(VBox(rows + [outw]))

    refresh_ui()

except Exception as e:
    print("Interactive UI not available in this environment:", e)
    print("Use list_pending_approvals() and approve_and_create(id, jira_tool) instead.")


Interactive UI not available in this environment: name 'approval_store' is not defined
Use list_pending_approvals() and approve_and_create(id, jira_tool) instead.


In [50]:
# Step 7A — Labeled evaluation dataset
# Each entry has: 
# - "text": a meeting snippet
# - "gold_actions": list of gold-standard action strings (simplified)

evaluation_dataset = [
    {
        "id": "ex1",
        "text": """Alice: TODO - Add unit tests for PaymentService.
                   Bob: Action item: update API docs.
                   Carol: This is general discussion, no action here.""",
        "gold_actions": [
            "Add unit tests for PaymentService",
            "update API docs"
        ]
    },
    {
        "id": "ex2",
        "text": """Team: We need to assign someone for auth migration.
                   TODO move login flow to new OAuth provider.
                   No action here, just FYI.""",
        "gold_actions": [
            "assign someone for auth migration",
            "move login flow to new OAuth provider"
        ]
    },
    {
        "id": "ex3",
        "text": """Bob: Reminder that Carol will prepare the migration script by Friday.
                   Carol: Action item: run load tests after deployment.""",
        "gold_actions": [
            "prepare the migration script by Friday",
            "run load tests after deployment"
        ]
    }
]

print("Evaluation dataset loaded with", len(evaluation_dataset), "examples.")


Evaluation dataset loaded with 3 examples.


In [51]:
# Step 7B — Evaluation harness
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

def normalize_text(s: str) -> str:
    return s.strip().lower()

def evaluate_extractor(controller: Controller, dataset, threshold=0.0):
    """
    Runs extractor on each example and computes P/R/F1 based on simple text matching.
    """
    y_true_all = []
    y_pred_all = []

    report = []

    for item in dataset:
        text = item["text"]
        gold = [normalize_text(g) for g in item["gold_actions"]]

        # Run pipeline (we only need extraction)
        res = controller.extractor.extract(
            controller.enricher.enrich(
                controller.normalizer.normalize(text)
            )
        )

        # Extract predicted action texts
        preds = [normalize_text(a["text"]) for a in res]

        # Build binary labels for P/R/F1
        # Any predicted action that contains (substring match) a gold action counts as correct
        matched = set()
        for p in preds:
            for g in gold:
                if g in p:
                    matched.add(g)

        # Build lists:
        # y_true: one entry per gold action
        # y_pred: predicted (1) or missed (0)
        for g in gold:
            y_true_all.append(1)
            y_pred_all.append(1 if g in matched else 0)

        # False positives: predictions that didn't match any gold
        false_pos = [p for p in preds if not any(g in p for g in gold)]

        report.append({
            "example_id": item["id"],
            "gold": gold,
            "preds": preds,
            "matched_gold": list(matched),
            "false_positives": false_pos
        })

    # Compute scores
    precision = precision_score(y_true_all, y_pred_all)
    recall = recall_score(y_true_all, y_pred_all)
    f1 = f1_score(y_true_all, y_pred_all)

    # Log into observability
    logger.info(json.dumps({
        "event": "evaluation_complete",
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "timestamp": time.time()
    }))

    return precision, recall, f1, report


In [52]:
precision, recall, f1, report = evaluate_extractor(ctrl, evaluation_dataset)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

print("\nDetailed confusion report:")
import pandas as pd
pd.DataFrame(report)


INFO:observability:{"event": "normalize", "segments": 3}


INFO:     127.0.0.1:58588 - "POST /repo/search HTTP/1.1" 200 OK


INFO:observability:{"event": "enrich", "repo_hits": 3, "memory_hits": 3}
INFO:observability:{"event": "extract", "actions_found": 2}
INFO:observability:{"event": "normalize", "segments": 3}


INFO:     127.0.0.1:58604 - "POST /repo/search HTTP/1.1" 200 OK


INFO:observability:{"event": "enrich", "repo_hits": 3, "memory_hits": 3}
INFO:observability:{"event": "extract", "actions_found": 1}
INFO:observability:{"event": "normalize", "segments": 2}


INFO:     127.0.0.1:58610 - "POST /repo/search HTTP/1.1" 200 OK


INFO:observability:{"event": "enrich", "repo_hits": 3, "memory_hits": 3}
INFO:observability:{"event": "extract", "actions_found": 1}
INFO:observability:{"event": "evaluation_complete", "precision": 1.0, "recall": 0.6666666666666666, "f1": 0.8, "timestamp": 1763257763.5183063}


Precision: 1.0
Recall: 0.6666666666666666
F1: 0.8

Detailed confusion report:


,example_id,gold,preds,matched_gold,false_positives
0,ex1,"[add unit tests for paymentservice, update api...",[alice: todo - add unit tests for paymentservi...,"[update api docs, add unit tests for paymentse...",[]
1,ex2,"[assign someone for auth migration, move login...",[todo move login flow to new oauth provider.],[move login flow to new oauth provider],[]
2,ex3,"[prepare the migration script by friday, run l...",[carol: action item: run load tests after depl...,[run load tests after deployment],[]


In [53]:
# Step 7D — expose evaluation scores as Prometheus metrics
from prometheus_client import Gauge

EVAL_PRECISION = Gauge("extractor_eval_precision", "Extractor precision score")
EVAL_RECALL = Gauge("extractor_eval_recall", "Extractor recall score")
EVAL_F1 = Gauge("extractor_eval_f1", "Extractor F1 score")

EVAL_PRECISION.set(precision)
EVAL_RECALL.set(recall)
EVAL_F1.set(f1)

print("Prometheus evaluation metrics exported.")


Prometheus evaluation metrics exported.


## 🧪 Agent Evaluation — Precision, Recall, F1

To quantify the quality of the ActionExtractor agent, we built a labeled dataset of real meeting snippets and computed:

- **Precision**
- **Recall**
- **F1 Score**

Each gold action item must match a predicted action. Substring matching is used to handle paraphrasing.

### Results

| Metric | Score |
|--------|-------|
| Precision | `{{precision}}` |
| Recall | `{{recall}}` |
| F1 Score | `{{f1}}` |

These scores are also exported to Prometheus at:

- `extractor_eval_precision`
- `extractor_eval_recall`
- `extractor_eval_f1`

Screenshots of the `/metrics` endpoint and Observability logs are included below.
